---
## 1. Install Libraries

In [ ]:
!pip install yfinance xgboost scikit-learn tensorflow matplotlib seaborn pandas numpy --quiet
print('Libraries installed.')

: 

---
## 2. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import yfinance as yf

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Fix random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

print('All libraries imported.')
print(f'  TensorFlow : {tf.__version__}')
print(f'  XGBoost    : {xgb.__version__}')

---
## 3. Configuration

In [ ]:
# Saudi stocks to test — matches the project stock list exactly
STOCKS = {
    '2222.SR': 'Saudi Aramco',
    '1120.SR': 'Al Rajhi Bank',
    '2010.SR': 'SABIC',
    '7010.SR': 'STC',
    '1180.SR': 'Bank Albilad',
    '1150.SR': 'Alinma Bank',
    '1010.SR': 'Riyad Bank',
    '1211.SR': 'Maaden',
    '2290.SR': 'Yansab',
    '2380.SR': 'Petro Rabigh',
    '4030.SR': 'Saudi Electricity',
    '2060.SR': 'Tasnee',
    '6010.SR': 'SAFCO',
    '1020.SR': 'Bank AlJazira',
    '7020.SR': 'Mobily',
}

START_DATE  = '2020-01-01'
END_DATE    = '2025-12-31'
TRAIN_SPLIT = 0.70   # 70% training
VAL_SPLIT   = 0.10   # 10% validation (hyperparameter tuning / early stopping)
# TEST_SPLIT = 0.20  # remaining 20% — held out, never seen during training
LSTM_WINDOW = 30     # look-back window (trading days)

FEATURES = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'MA_7', 'MA_21', 'MA_50',
    'STD_7', 'STD_21',
    'RSI', 'MACD', 'MACD_signal',
    'BB_upper', 'BB_lower', 'BB_width',
    'Return_1d', 'Return_5d',
    'High_Low_Range', 'Volume_Ratio'
]

print(f'Stocks      : {len(STOCKS)}')
print(f'Features    : {len(FEATURES)}')
print(f'Period      : {START_DATE} to {END_DATE}')
print(f'Split       : {int(TRAIN_SPLIT*100)}% train / {int(VAL_SPLIT*100)}% val / {int((1-TRAIN_SPLIT-VAL_SPLIT)*100)}% test')

---
## 4. Helper Functions

In [ ]:
def fetch_stock(ticker, start, end):
    """Download OHLCV data from Yahoo Finance."""
    raw = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    return raw[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()


def add_features(df):
    """Compute technical indicators used as model features."""
    d = df.copy()

    # Moving averages
    d['MA_7']  = d['Close'].rolling(7).mean()
    d['MA_21'] = d['Close'].rolling(21).mean()
    d['MA_50'] = d['Close'].rolling(50).mean()

    # Rolling standard deviation (volatility proxy)
    d['STD_7']  = d['Close'].rolling(7).std()
    d['STD_21'] = d['Close'].rolling(21).std()

    # RSI (14-day)
    delta = d['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    d['RSI'] = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

    # MACD
    ema12 = d['Close'].ewm(span=12).mean()
    ema26 = d['Close'].ewm(span=26).mean()
    d['MACD']        = ema12 - ema26
    d['MACD_signal'] = d['MACD'].ewm(span=9).mean()

    # Bollinger Bands (20-day)
    mid = d['Close'].rolling(20).mean()
    std = d['Close'].rolling(20).std()
    d['BB_upper'] = mid + 2 * std
    d['BB_lower'] = mid - 2 * std
    d['BB_width'] = d['BB_upper'] - d['BB_lower']

    # Price returns
    d['Return_1d'] = d['Close'].pct_change(1)
    d['Return_5d'] = d['Close'].pct_change(5)

    # Intraday range relative to close
    d['High_Low_Range'] = (d['High'] - d['Low']) / d['Close']

    # Volume relative to 7-day average
    d['Volume_Ratio'] = d['Volume'] / d['Volume'].rolling(7).mean()

    # Target: next-day closing price
    d['Target'] = d['Close'].shift(-1)

    return d.dropna()


def make_sequences(X, y, window):
    """Convert flat arrays into (samples, window, features) tensors for LSTM."""
    Xs, ys = [], []
    for i in range(window, len(X)):
        Xs.append(X[i - window:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)


def calc_metrics(model_name, ticker, y_true, y_pred):
    """Return MAE, RMSE, R2, and Accuracy for a single model/stock pair."""
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))) * 100
    return {
        'Model'   : model_name,
        'Ticker'  : ticker,
        'MAE'     : mean_absolute_error(y_true, y_pred),
        'RMSE'    : np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2'      : r2_score(y_true, y_pred),
        'Accuracy': round(max(0.0, 100 - mape), 2),
    }


def build_lstm(n_features, window):
    """Build and compile a stacked LSTM model."""
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=(window, n_features)),
        Dropout(0.2),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


print('Helper functions ready.')

---
## 5. Download Saudi Stock Data from Yahoo Finance

In [ ]:
stocks_data = {}

for ticker, name in STOCKS.items():
    df = fetch_stock(ticker, START_DATE, END_DATE)
    if df.empty:
        print(f'  WARNING: {ticker} ({name}) — no data returned.')
        continue
    stocks_data[ticker] = df
    print(f'  OK  {ticker} ({name:<15})  {len(df):>4} trading days  '
          f'{df.index[0].date()} to {df.index[-1].date()}')

print(f'\nTotal stocks loaded: {len(stocks_data)}')

---
## 6. Exploratory Data Analysis (EDA)

In [ ]:
# Generate enough colors for all stocks using a colormap
cmap   = plt.cm.tab20
colors = [cmap(i / max(len(stocks_data) - 1, 1)) for i in range(len(stocks_data))]

fig, axes = plt.subplots(len(stocks_data), 1,
                          figsize=(14, 4 * len(stocks_data)),
                          sharex=False)
if len(stocks_data) == 1:
    axes = [axes]

for ax, (ticker, df), color in zip(axes, stocks_data.items(), colors):
    name = STOCKS[ticker]
    ax.plot(df.index, df['Close'], color=color, linewidth=1.4)
    ax.fill_between(df.index, df['Close'], alpha=0.08, color=color)
    ax.set_title(f'{name}  ({ticker})', fontweight='bold', fontsize=12)
    ax.set_ylabel('Price (SAR)')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle(f'Closing Prices — Saudi Market {START_DATE[:4]}-{END_DATE[:4]}  ({len(stocks_data)} stocks)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    STOCKS[t]: df['Close'].describe().round(2)
    for t, df in stocks_data.items()
})
print('\n=== Descriptive Statistics — Close Price ===')
display(summary)

---
## 7. Train All Three Algorithms on Every Stock

> **Three-way chronological split — 70% Train / 10% Validation / 20% Test**
> - **Train**: model learns from this data only
> - **Validation**: used for early stopping — never influences final metrics
> - **Test**: held-out set, evaluated once at the end — no data leakage

> **Random Forest → XGBoost → LSTM** are each trained and evaluated per stock.

In [ ]:
all_results = []
predictions = {}

early_stop = EarlyStopping(
    monitor='val_loss', patience=8,
    restore_best_weights=True, verbose=0
)

# Build LSTM once to avoid TensorFlow retracing inside the loop
shared_lstm     = build_lstm(len(FEATURES), LSTM_WINDOW)
initial_weights = shared_lstm.get_weights()

for ticker, df_raw in stocks_data.items():
    name = STOCKS[ticker]
    print(f'\n{"="*55}')
    print(f'  {name} ({ticker})')
    print(f'{"="*55}')

    df    = add_features(df_raw)
    X     = df[FEATURES].values
    y     = df['Target'].values
    dates = df.index
    n     = len(X)

    # Chronological three-way split — no shuffling
    train_end = int(n * TRAIN_SPLIT)
    val_end   = int(n * (TRAIN_SPLIT + VAL_SPLIT))

    X_tr_raw  = X[:train_end]
    X_val_raw = X[train_end:val_end]
    X_te_raw  = X[val_end:]

    y_tr  = y[:train_end]
    y_val = y[train_end:val_end]
    y_te  = y[val_end:]
    dates_te = dates[val_end:]

    # Fit scalers on TRAINING data only — no data leakage
    scaler_X = MinMaxScaler().fit(X_tr_raw)
    scaler_y = MinMaxScaler().fit(y_tr.reshape(-1, 1))

    X_tr  = scaler_X.transform(X_tr_raw)
    X_val = scaler_X.transform(X_val_raw)
    X_te  = scaler_X.transform(X_te_raw)

    # --- Random Forest ---
    print('  [1/3] Random Forest ...')
    rf = RandomForestRegressor(
        n_estimators=300, max_depth=10,
        min_samples_split=5, n_jobs=-1, random_state=42
    )
    rf.fit(X_tr, y_tr)
    rf_pred = rf.predict(X_te)
    res = calc_metrics('Random Forest', ticker, y_te, rf_pred)
    all_results.append(res)
    print(f'         Accuracy={res["Accuracy"]:.2f}%  MAE={res["MAE"]:.3f}  RMSE={res["RMSE"]:.3f}  R2={res["R2"]:.4f}')

    # --- XGBoost (uses validation set for early stopping) ---
    print('  [2/3] XGBoost ...')
    xgb_m = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgb_m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    xgb_pred = xgb_m.predict(X_te)
    res = calc_metrics('XGBoost', ticker, y_te, xgb_pred)
    all_results.append(res)
    print(f'         Accuracy={res["Accuracy"]:.2f}%  MAE={res["MAE"]:.3f}  RMSE={res["RMSE"]:.3f}  R2={res["R2"]:.4f}')

    # --- LSTM (uses explicit validation set for early stopping) ---
    print('  [3/3] LSTM ...')
    shared_lstm.set_weights(initial_weights)

    X_all_sc = scaler_X.transform(X)
    y_all_sc = scaler_y.transform(y.reshape(-1, 1)).ravel()

    X_seq, y_seq = make_sequences(X_all_sc, y_all_sc, LSTM_WINDOW)

    # Map original split indices to sequence indices
    ltr_end  = train_end - LSTM_WINDOW
    lval_end = val_end   - LSTM_WINDOW

    X_ltr,  y_ltr  = X_seq[:ltr_end],           y_seq[:ltr_end]
    X_lval, y_lval = X_seq[ltr_end:lval_end],    y_seq[ltr_end:lval_end]
    X_lte,  y_lte  = X_seq[lval_end:],           y_seq[lval_end:]

    y_lte_actual = scaler_y.inverse_transform(y_lte.reshape(-1, 1)).ravel()
    dates_lte    = dates[val_end:]

    shared_lstm.fit(
        X_ltr, y_ltr,
        validation_data=(X_lval, y_lval),
        epochs=80, batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )
    lstm_pred_sc = shared_lstm.predict(X_lte, verbose=0)
    lstm_pred    = scaler_y.inverse_transform(lstm_pred_sc).ravel()
    res = calc_metrics('LSTM', ticker, y_lte_actual, lstm_pred)
    all_results.append(res)
    print(f'         Accuracy={res["Accuracy"]:.2f}%  MAE={res["MAE"]:.3f}  RMSE={res["RMSE"]:.3f}  R2={res["R2"]:.4f}')

    predictions[ticker] = {
        'dates_te' : dates_te,
        'y_te'     : y_te,
        'rf_pred'  : rf_pred,
        'xgb_pred' : xgb_pred,
        'dates_lte': dates_lte,
        'y_lte'    : y_lte_actual,
        'lstm_pred': lstm_pred,
    }

print(f'\nTraining complete. Total results: {len(all_results)}')

---
## 8. Detailed Results Table (Stock x Algorithm)

In [ ]:
results_df = pd.DataFrame(all_results)
results_df['Stock'] = results_df['Ticker'].map(STOCKS)

pivot_acc  = results_df.pivot(index='Stock', columns='Model', values='Accuracy').round(2)
pivot_r2   = results_df.pivot(index='Stock', columns='Model', values='R2').round(4)
pivot_rmse = results_df.pivot(index='Stock', columns='Model', values='RMSE').round(4)

print('=== Accuracy % per Stock per Algorithm (higher is better) ===')
display(pivot_acc.style.background_gradient(cmap='Greens', axis=1))

print('\n=== R2 per Stock per Algorithm (higher is better) ===')
display(pivot_r2.style.background_gradient(cmap='Blues', axis=1))

print('\n=== RMSE per Stock per Algorithm (lower is better) ===')
display(pivot_rmse.style.background_gradient(cmap='YlOrRd_r', axis=1))

---
## 9. Overall Comparison — Average Across All 15 Stocks

In [ ]:
avg = results_df.groupby('Model')[['MAE', 'RMSE', 'R2', 'Accuracy']].mean().round(4)
avg['Accuracy'] = avg['Accuracy'].round(2)

print(f'=== Average Performance Across {len(stocks_data)} Saudi Stocks ===')
display(avg)

model_colors = {
    'Random Forest': '#10B981',
    'XGBoost'      : '#F59E0B',
    'LSTM'         : '#2563EB'
}
bar_colors = [model_colors[m] for m in avg.index]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, metric in zip(axes, ['Accuracy', 'MAE', 'RMSE', 'R2']):
    vals = avg[metric]
    bars = ax.bar(vals.index, vals.values, color=bar_colors,
                  edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, vals.values):
        label = f'{val:.2f}%' if metric == 'Accuracy' else f'{val:.4f}'
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + vals.max() * 0.01,
                label, ha='center', va='bottom',
                fontsize=10, fontweight='bold')
    ax.set_title(f'Average {metric}', fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='y', alpha=0.3)
    if metric in ('R2', 'Accuracy'):
        ax.set_ylim(0, max(vals.max() * 1.15, 1.1 if metric == 'R2' else 105))

plt.suptitle(f'Algorithm Comparison — Average over {len(stocks_data)} Saudi Stocks',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 10. Heatmap — R2 and RMSE per Stock

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

sns.heatmap(pivot_acc, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, ax=axes[0], vmin=0, vmax=100)
axes[0].set_title('Accuracy %  (higher is better)', fontweight='bold')

sns.heatmap(pivot_r2, annot=True, fmt='.4f', cmap='RdYlGn',
            linewidths=0.5, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('R2  (higher is better)', fontweight='bold')

sns.heatmap(pivot_rmse, annot=True, fmt='.4f', cmap='RdYlGn_r',
            linewidths=0.5, ax=axes[2])
axes[2].set_title('RMSE  (lower is better)', fontweight='bold')

plt.suptitle('Performance Heatmap — Stock x Algorithm',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 11. Predictions vs. Actual Values (per Stock)

In [ ]:
for ticker, preds in predictions.items():
    stock_name = STOCKS[ticker]
    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)

    pairs = [
        ('Random Forest', preds['rf_pred'],   preds['y_te'],  preds['dates_te'],  '#10B981'),
        ('XGBoost',       preds['xgb_pred'],  preds['y_te'],  preds['dates_te'],  '#F59E0B'),
        ('LSTM',          preds['lstm_pred'], preds['y_lte'], preds['dates_lte'], '#2563EB'),
    ]

    for ax, (model_name, pred, actual, dates, color) in zip(axes, pairs):
        n  = min(len(pred), len(actual), len(dates))
        r2 = r2_score(actual[:n], pred[:n])
        ax.plot(dates[:n], actual[:n], color='#374151', lw=1.5,
                label='Actual', alpha=0.85)
        ax.plot(dates[:n], pred[:n], color=color, lw=1.5,
                label=f'{model_name} prediction', linestyle='--')
        ax.set_title(f'{model_name}  —  R2 = {r2:.4f}', fontweight='bold')
        ax.set_ylabel('Price (SAR)')
        ax.legend(loc='upper left', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

    plt.suptitle(f'Predictions vs Actual — {stock_name} ({ticker})',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

---
## 12. Scatter Plots — Prediction Accuracy

In [ ]:
for ticker, preds in predictions.items():
    stock_name = STOCKS[ticker]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    scatter_pairs = [
        ('Random Forest', preds['rf_pred'],   preds['y_te'],  '#10B981'),
        ('XGBoost',       preds['xgb_pred'],  preds['y_te'],  '#F59E0B'),
        ('LSTM',          preds['lstm_pred'], preds['y_lte'], '#2563EB'),
    ]

    for ax, (model_name, pred, actual, color) in zip(axes, scatter_pairs):
        n  = min(len(pred), len(actual))
        r2 = r2_score(actual[:n], pred[:n])
        ax.scatter(actual[:n], pred[:n], alpha=0.4, color=color, s=15)
        mn, mx = actual[:n].min(), actual[:n].max()
        ax.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect fit')
        ax.set_title(f'{model_name}\nR2 = {r2:.4f}', fontweight='bold')
        ax.set_xlabel('Actual Price')
        ax.set_ylabel('Predicted Price')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.suptitle(f'Scatter — {stock_name} ({ticker})',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

---
## 13. Final Ranking — Best Algorithm

---
## 14. Winner Model Verification — XGBoost

In [ ]:
# Rank models and pick winner
final  = avg.sort_values(['Accuracy', 'R2'], ascending=False)
winner = final.index[0]

# Map winner name to prediction key in the predictions dict
PRED_KEY = {'XGBoost': 'xgb_pred', 'Random Forest': 'rf_pred', 'LSTM': 'lstm_pred'}
Y_KEY    = {'XGBoost': 'y_te',     'Random Forest': 'y_te',    'LSTM': 'y_lte'}
pred_key = PRED_KEY[winner]
y_key    = Y_KEY[winner]

winner_results = results_df[results_df['Model'] == winner].copy()

# --- 1. Per-stock Accuracy ---
print(f'=== {winner} — Accuracy per Stock ===')
display(
    winner_results[['Stock', 'Accuracy', 'MAE', 'RMSE', 'R2']]
    .sort_values('Accuracy', ascending=False)
    .reset_index(drop=True)
)

acc = winner_results['Accuracy']
print(f'\nMean   : {acc.mean():.2f}%')
print(f'Best   : {acc.max():.2f}%  — {winner_results.loc[acc.idxmax(), "Stock"]}')
print(f'Worst  : {acc.min():.2f}%  — {winner_results.loc[acc.idxmin(), "Stock"]}')
print(f'StdDev : {acc.std():.2f}%  (lower = more consistent across stocks)')

# --- 2. Directional Accuracy ---
dir_rows = []
for ticker, preds in predictions.items():
    actual = preds[y_key]
    pred   = preds[pred_key]
    n      = min(len(actual), len(pred)) - 1
    if n < 1:
        continue
    correct = np.mean((np.diff(actual[:n+1]) > 0) == (np.diff(pred[:n+1]) > 0)) * 100
    dir_rows.append({'Stock': STOCKS[ticker], 'Directional Accuracy %': round(correct, 2)})

dir_df = pd.DataFrame(dir_rows).sort_values('Directional Accuracy %', ascending=False)
print(f'\n=== {winner} — Directional Accuracy (correct up/down prediction) ===')
display(dir_df.reset_index(drop=True))
print(f'Average Directional Accuracy: {dir_df["Directional Accuracy %"].mean():.2f}%')
print('(Baseline random guess = 50%)')

# --- 3. Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sorted_acc = winner_results.sort_values('Accuracy')
bar_colors = ['#10B981' if a >= 90 else '#F59E0B' if a >= 80 else '#EF4444'
              for a in sorted_acc['Accuracy']]
bars = axes[0].barh(sorted_acc['Stock'], sorted_acc['Accuracy'], color=bar_colors)
axes[0].axvline(x=90, color='red', linestyle='--', alpha=0.5, label='90% threshold')
axes[0].set_title(f'{winner} — Accuracy per Stock', fontweight='bold')
axes[0].set_xlabel('Accuracy %')
axes[0].set_xlim(0, 108)
axes[0].legend()
for bar, val in zip(bars, sorted_acc['Accuracy']):
    axes[0].text(val + 0.3, bar.get_y() + bar.get_height() / 2,
                 f'{val:.2f}%', va='center', fontsize=9)

sorted_dir = dir_df.sort_values('Directional Accuracy %')
dir_colors = ['#10B981' if a >= 55 else '#EF4444' for a in sorted_dir['Directional Accuracy %']]
bars2 = axes[1].barh(sorted_dir['Stock'], sorted_dir['Directional Accuracy %'], color=dir_colors)
axes[1].axvline(x=50, color='red', linestyle='--', alpha=0.5, label='Random baseline (50%)')
axes[1].set_title(f'{winner} — Directional Accuracy per Stock', fontweight='bold')
axes[1].set_xlabel('Directional Accuracy %')
axes[1].set_xlim(0, 108)
axes[1].legend()
for bar, val in zip(bars2, sorted_dir['Directional Accuracy %']):
    axes[1].text(val + 0.3, bar.get_y() + bar.get_height() / 2,
                 f'{val:.2f}%', va='center', fontsize=9)

plt.suptitle(f'{winner} — Model Verification', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Rank by Accuracy (primary) then R2 (secondary)
final = avg.sort_values(['Accuracy', 'R2'], ascending=False)

print('=' * 60)
print(f'   Final Ranking -- Average over {len(stocks_data)} Saudi Stocks')
print('=' * 60)
display(final)

winner    = final.index[0]
runner_up = final.index[1] if len(final) > 1 else '-'
baseline  = final.index[2] if len(final) > 2 else '-'

print()
print(f'  Winner (best) : {winner}  —  Accuracy: {final.loc[winner, "Accuracy"]:.2f}%')
print(f'  Runner-up     : {runner_up}  —  Accuracy: {final.loc[runner_up, "Accuracy"]:.2f}%')
print(f'  Baseline      : {baseline}  —  Accuracy: {final.loc[baseline, "Accuracy"]:.2f}%')
print()

notes = {
    'Random Forest': 'Fast, stable, easy to deploy -- strong production baseline.',
    'XGBoost'      : 'Best accuracy on tabular financial data with interpretable feature importance.',
    'LSTM'         : 'Captures long-term temporal dependencies -- theoretically strongest for time series.',
}
print(f'Why {winner}?')
print(notes.get(winner, ''))
print()
print(f'{runner_up} and {baseline} serve as comparison baselines.')

---
## 14. Export the Winning Model

In [ ]:
import joblib, os

os.makedirs('exported_model', exist_ok=True)

# Train the winner on ALL stocks combined for a more general model
all_X, all_y = [], []
for ticker, df_raw in stocks_data.items():
    df_exp = add_features(df_raw)
    all_X.append(df_exp[FEATURES].values)
    all_y.append(df_exp['Target'].values)

X_combined = np.vstack(all_X)
y_combined  = np.concatenate(all_y)

scaler_X_exp  = MinMaxScaler()
X_combined_sc = scaler_X_exp.fit_transform(X_combined)

joblib.dump(scaler_X_exp, 'exported_model/scaler_X.pkl')
print('scaler_X saved.')

if winner == 'Random Forest':
    final_model = RandomForestRegressor(
        n_estimators=300, max_depth=10,
        min_samples_split=5, n_jobs=-1, random_state=42
    )
    final_model.fit(X_combined_sc, y_combined)
    joblib.dump(final_model, 'exported_model/best_model.pkl')
    print('Random Forest saved -> exported_model/best_model.pkl')

elif winner == 'XGBoost':
    final_model = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        random_state=42, n_jobs=-1, verbosity=0
    )
    final_model.fit(X_combined_sc, y_combined)
    joblib.dump(final_model, 'exported_model/best_model.pkl')
    print('XGBoost saved -> exported_model/best_model.pkl')

elif winner == 'LSTM':
    X_seq, y_seq_raw = make_sequences(X_combined_sc, y_combined, LSTM_WINDOW)
    lstm_exp = build_lstm(len(FEATURES), LSTM_WINDOW)
    lstm_exp.fit(X_seq, y_seq_raw, epochs=80, batch_size=32,
                 validation_split=0.1,
                 callbacks=[EarlyStopping(monitor='val_loss', patience=8,
                                          restore_best_weights=True, verbose=0)],
                 verbose=0)
    lstm_exp.save('exported_model/best_model.keras')
    joblib.dump({'window': LSTM_WINDOW}, 'exported_model/best_model.pkl')
    print('LSTM saved -> exported_model/best_model.keras')

print(f'\nWinner deployed: {winner}')
print('\nDownloading files...')
from google.colab import files
files.download('exported_model/best_model.pkl')
files.download('exported_model/scaler_X.pkl')
print('Done.')

---
## 15. Integration — Flask API + Investment.tsx

### Backend — `predict_api.py`
```python
from flask import Flask, request, jsonify
from flask_cors import CORS
import joblib, numpy as np, yfinance as yf

app = Flask(__name__)
CORS(app)

model    = joblib.load('exported_model/best_model.pkl')   # .json for XGBoost
scaler_X = joblib.load('exported_model/scaler_X.pkl')

@app.route('/predict', methods=['POST'])
def predict():
    ticker = request.json.get('ticker', '2222.SR')
    df     = yf.download(ticker, period='6mo', auto_adjust=True, progress=False)
    # ... add_features(df) ...
    row  = df[FEATURES].iloc[[-1]].values
    pred = model.predict(scaler_X.transform(row))
    return jsonify({'ticker': ticker, 'predicted_price': round(float(pred[0]), 2)})

if __name__ == '__main__':
    app.run(port=5000)
```

### Frontend — `Investment.tsx`
```ts
const getPrediction = async (ticker: string) => {
  const res = await fetch('http://localhost:5000/predict', {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify({ ticker })
  });
  const { predicted_price } = await res.json();
  return predicted_price;   // next-day predicted close
};
```